# residual-skip-add — worked example 1: Pre-activation residual block add

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `residual-skip-add`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A residual block computes `out = F(x) + shortcut(x)`. When the residual branch keeps channels and spatial size unchanged, the shortcut is the identity, so the add is a literal `F(x) + x`. The skip connection creates a gradient highway: gradients flow straight through the addition to the input.

## Worked solution

We build a small block with two stacked 3x3 convolutions (padding 1, stride 1) as the residual function `F`. Because in/out channels match and stride is 1, the shortcut is `nn.Identity`, so forward returns `self.f(x) + self.skip(x)` which equals `F(x) + x`. We construct the block, push a `(2, 8, 16, 16)` input through, and confirm the output keeps the same shape — the defining property of an identity-shortcut residual block. We print the output shape to show the spatial/channel dimensions are preserved by the add.

In [ ]:
import torch.nn as nn
Tensor = t.Tensor


class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
        self.skip = nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
block = ResBlock(8)
x = t.randn(2, 8, 16, 16)
out = block(x)
print('input shape: ', tuple(x.shape))
print('output shape:', tuple(out.shape))
print('shape preserved:', x.shape == out.shape)